<table>
 <tr align=left><td><img align=left src="https://i.creativecommons.org/l/by/4.0/88x31.png">
 <td>Text provided under a Creative Commons Attribution license, CC-BY. All code is made available under the FSF-approved MIT license. (c) Kyle T. Mandli</td>
</table>

# GeoClaw Meteorological Forcing Tutorial

**Parametric (Holland) vs. gridded (NetCDF) wind and pressure forcing.**

This notebook is a short, self-contained tour of GeoClaw's two meteorological
forcing families, selected through the `storm_family` / `storm_subtype` API on
`rundata.surge_data`:

1. **Parametric** &mdash; an analytic Holland 1980 storm built from a track.
2. **Gridded** &mdash; wind/pressure fields read from a CF-compliant NetCDF file
   (requires building GeoClaw with `-DNETCDF`).

It uses a deliberately tiny idealized domain (20&times;20 cells, flat
bathymetry, no AMR, no downloads) so it runs in a few minutes.  See the GeoClaw
documentation section *Meteorological (Storm) Forcing* for background, and the
`katrina` notebook for a full real-storm workflow.

*Version: runs against a current Clawpack/GeoClaw install.*

In [ ]:
%matplotlib inline
from __future__ import print_function

## Setup

We import the notebook build helpers (`nbtools`), GeoClaw's topography and storm
tools, and `IPython.display.Image` for inline figures.  The wind (`wind_u`,
`wind_v`) and pressure fields live in the GeoClaw `aux` array and are written
out each frame so we can plot them.

In [ ]:
import os
from pathlib import Path
import numpy as np

from clawpack.clawutil import nbtools
import clawpack.geoclaw.topotools as topotools
import clawpack.geoclaw.surge.storm as storm

from IPython.display import Image

from setrun import setrun

## A flat idealized ocean

Both runs share a flat 200 m deep basin over `lon = [-5, 5]`, `lat = [15, 25]`.
Writing it as a topotype-3 file is all the bathymetry we need.

In [ ]:
topo = topotools.Topography(topo_func=lambda x, y: -200.0 + 0.0 * x)
topo.topo_type = 3
topo.x = np.linspace(-6.0, 6.0, 25)
topo.y = np.linspace(14.0, 26.0, 25)
topo.write("flat.tt3", topo_type=3, Z_format="%22.15e")
print("wrote flat.tt3")

## Part 1 &mdash; Parametric Holland storm

We build a short 4-point storm track (eye drifting across the domain, 50 m/s
maximum winds, 950 mbar central pressure) and write it as a GeoClaw storm file.
This is exactly what `Storm.write(..., file_format="geoclaw")` produces from any
ingested track (ATCF, HURDAT, ...).

In [ ]:
TIME_OFFSET = np.datetime64("2020-08-01T00:00:00")
hours = np.array([0.0, 3.0, 6.0, 9.0])
n = len(hours)

s = storm.Storm()
s.t = TIME_OFFSET + (hours * 3600.0).astype("timedelta64[s]")
s.time_offset = TIME_OFFSET
s.eye_location = np.empty((n, 2))
s.eye_location[:, 0] = np.linspace(0.0, 1.0, n)      # lon
s.eye_location[:, 1] = np.linspace(20.0, 20.5, n)    # lat
s.max_wind_speed = np.full(n, 50.0)                  # m/s
s.max_wind_radius = np.full(n, 50.0e3)               # m
s.central_pressure = np.full(n, 95000.0)             # Pa
s.storm_radius = np.full(n, 300.0e3)                 # m
s.write("holland.storm", file_format="geoclaw")
print("wrote holland.storm")

The parametric family is selected in `setrun.py` with:

```python
surge_data.storm_family  = "parametric"
surge_data.storm_subtype = "holland80"
surge_data.storm_file    = "holland.storm"
```

(The legacy `surge_data.storm_specification_type = "holland80"` selects the same
forcing and is still supported.)

Compile a plain `xgeoclaw` (no NetCDF needed for the parametric path):

In [ ]:
nbtools.make_exe(new=True, verbose=False)

Write the data files for the Holland run and run the simulation, sending
output/plots to labeled directories so we can keep both runs side by side.

In [ ]:
setrun(forcing="holland80").write()
outdir_h, plotdir_h = nbtools.make_output_and_plots(label="holland", verbose=False)
print("output:", outdir_h, "| plots:", plotdir_h)

The wind vortex, the pressure low, and the resulting surface response at
the final frame (6 h), plus the surface time series at the gauge under the
track:

In [ ]:
for figno, caption in [(1, "wind speed"), (2, "sea-level pressure"),
                       (0, "surface elevation")]:
    display(Image(os.path.join(plotdir_h, "frame0002fig%d.png" % figno), width=460))
display(Image(os.path.join(plotdir_h, "gauge0001fig300.png"), width=460))

## Part 2 &mdash; Gridded NetCDF forcing

Gridded forcing reads wind/pressure from external field files.  GeoClaw must be
compiled with `-DNETCDF` and linked against netcdf-fortran; we probe `nf-config`
for the flags and skip this part gracefully if it (or `xarray`) is unavailable.

In [ ]:
import shutil, subprocess

def _netcdf_flags():
    nf, nc = shutil.which("nf-config"), shutil.which("nc-config")
    if nf is None:
        return None
    try:
        fflags = subprocess.check_output([nf, "--fflags"], text=True).strip()
        flibs = subprocess.check_output([nf, "--flibs"], text=True).strip()
        if nc is not None:
            flibs += " " + subprocess.check_output([nc, "--libs"], text=True).strip()
    except (subprocess.CalledProcessError, OSError):
        return None
    return fflags, flibs

try:
    import xarray  # noqa: F401
    import netCDF4  # noqa: F401
    _have_xarray = True
except ImportError:
    _have_xarray = False

_flags = _netcdf_flags()
NETCDF_AVAILABLE = (_flags is not None) and _have_xarray
if NETCDF_AVAILABLE:
    fflags, flibs = _flags
    netcdf_env = dict(os.environ, USE_NETCDF="1",
                      NETCDF_FFLAGS=fflags, NETCDF_LFLAGS=flibs)
    print("NetCDF available -- will build with -DNETCDF and run the gridded case.")
else:
    netcdf_env = None
    print("NetCDF (nf-config / xarray) unavailable -- skipping the gridded part.")

Build a small CF-compliant NetCDF forcing file: a drifting Gaussian
vortex of eastward/northward wind (`u10`, `v10`) and mean-sea-level pressure
(`msl`), mirroring the parametric storm.  GeoClaw discovers the variables and
coordinates from their CF metadata.  `Storm.write(..., file_format="data")`
then writes the small `.storm` descriptor that `surge_data.storm_file` points
at.

In [ ]:
if NETCDF_AVAILABLE:
    import xarray as xr

    lon = np.linspace(-5.0, 5.0, 21)
    lat = np.linspace(15.0, 25.0, 21)
    time = TIME_OFFSET + (hours * 3600.0).astype("timedelta64[s]")
    LON, LAT = np.meshgrid(lon, lat, indexing="xy")

    u = np.empty((n, len(lat), len(lon)))
    v = np.empty_like(u)
    p = np.empty_like(u)
    for k in range(n):
        cx = 0.0 + (1.0 / 3.0) * (hours[k] / 3.0)
        cy = 20.0 + (0.5 / 3.0) * (hours[k] / 3.0)
        env = np.exp(-((LON - cx) ** 2 + (LAT - cy) ** 2) / 4.0)
        u[k] = -20.0 * (LAT - cy) * env
        v[k] = 20.0 * (LON - cx) * env
        p[k] = 101300.0 - 6000.0 * env

    ds = xr.Dataset(
        {
            "u10": (("valid_time", "latitude", "longitude"), u,
                    {"units": "m/s", "standard_name": "eastward_wind"}),
            "v10": (("valid_time", "latitude", "longitude"), v,
                    {"units": "m/s", "standard_name": "northward_wind"}),
            "msl": (("valid_time", "latitude", "longitude"), p,
                    {"units": "Pa",
                     "standard_name": "air_pressure_at_mean_sea_level"}),
        },
        coords={
            "longitude": ("longitude", lon, {"units": "degrees_east", "axis": "X"}),
            "latitude": ("latitude", lat, {"units": "degrees_north", "axis": "Y"}),
            "valid_time": ("valid_time", time, {"axis": "T"}),
        },
    )
    ds.to_netcdf("met.nc")

    g = storm.Storm()
    g.time_offset = TIME_OFFSET
    g.file_format = "netcdf"
    # Absolute path: GeoClaw runs from the output directory and the descriptor
    # records where to find the .nc file.  write_data expects pathlib.Path.
    g.file_paths = [Path("met.nc").resolve()]
    g.write(Path("met.storm"), file_format="data")
    print("wrote met.nc and met.storm")

Rebuild with NetCDF support, then run the gridded case.  The only change
in `setrun.py` is the forcing selection:

```python
surge_data.storm_family  = "gridded"
surge_data.storm_subtype = "gridded"
surge_data.storm_file    = "met.storm"
```

In [ ]:
if NETCDF_AVAILABLE:
    nbtools.make_exe(new=True, env=netcdf_env, verbose=False)
    setrun(forcing="data").write()
    outdir_n, plotdir_n = nbtools.make_output_and_plots(label="netcdf", verbose=False)
    print("output:", outdir_n, "| plots:", plotdir_n)

In [ ]:
if NETCDF_AVAILABLE:
    for figno in (1, 2, 0):
        display(Image(os.path.join(plotdir_n, "frame0002fig%d.png" % figno), width=460))
    display(Image(os.path.join(plotdir_n, "gauge0001fig300.png"), width=460))

## Summary

The same simulation was forced two ways, chosen entirely through
`surge_data.storm_family` / `storm_subtype`:

| | family | subtype | storm_file |
|---|---|---|---|
| Parametric | `parametric` | `holland80` | GeoClaw storm file |
| Gridded | `gridded` | `gridded` | NetCDF descriptor |

The parametric storm is generated analytically from a compact track; the gridded
case interpolates external NetCDF fields (and needs a `-DNETCDF` build).  For the
variable/coordinate conventions GeoClaw expects in a NetCDF forcing file, and for
OWI/ASCII forcing, see the *NetCDF input* and *Meteorological (Storm) Forcing*
documentation.  For a full real-storm surge workflow (downloaded ATCF track and
bathymetry, AMR, tide-gauge comparison), see the `katrina` notebook.